In [0]:
df = spark.read.table("Weather_Analytics.silver.silver_weather_clean")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
Q1 = spark.sql("""
SELECT
    city,
    event_time AS latest_event_time,
    temperature AS latest_temperature,
    humidity AS latest_humidity,
    weather_condition AS latest_weather_condition,
    rainfall AS latest_rainfall,

    CASE
        WHEN weather_severity_score BETWEEN 0 AND 1 THEN 'Low'
        WHEN weather_severity_score BETWEEN 2 AND 3 THEN 'Medium'
        WHEN weather_severity_score BETWEEN 4 AND 5 THEN 'High'
        ELSE 'Critical'
    END AS risk_level

FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY city
               ORDER BY event_time DESC, ingestion_time DESC
           ) AS rn
    FROM Weather_Analytics.silver.silver_weather_clean
    WHERE city IS NOT NULL
      AND event_time IS NOT NULL
) t
WHERE rn = 1
""")

Q1.display()

Q1.write.mode("overwrite").saveAsTable("Weather_Analytics.gold.gold_city_weather")

city,latest_event_time,latest_temperature,latest_humidity,latest_weather_condition,latest_rainfall,risk_level
Bangalore,2026-03-08T23:00:00.000Z,30.2,53,Thunderstorm,4.1,Low
Chennai,2026-03-08T23:00:00.000Z,31.8,61,Partly Cloudy,6.8,Low
Delhi,2026-03-08T23:00:00.000Z,39.0,42,Rainy,2.1,Low
Hyderabad,2026-03-08T23:00:00.000Z,27.1,63,Haze,11.1,Low
Mumbai,2026-03-08T23:00:00.000Z,28.2,71,Clear,3.5,Low
Pune,2026-03-08T23:00:00.000Z,29.3,72,Rainy,5.7,Low


In [0]:
Q2 = spark.sql("""
WITH latest_date AS (
    SELECT MAX(event_date) AS max_date
    FROM Weather_Analytics.silver.silver_weather_clean
),

filtered AS (
    SELECT *
    FROM Weather_Analytics.silver.silver_weather_clean
    WHERE event_date >= (SELECT max_date - INTERVAL 7 DAYS FROM latest_date)
      AND temperature > 40
)

SELECT
    city,
    COUNT(*) AS heatwave_occurrences,
    MIN(event_time) AS first_occurrence,
    MAX(event_time) AS latest_occurrence,

    CASE
        WHEN MAX(event_time) >= (SELECT max_date FROM latest_date) - INTERVAL 1 DAY THEN 'Active'
        ELSE 'Monitoring'
    END AS alert_status

FROM filtered
GROUP BY city
HAVING COUNT(*) >= 3
""")

Q2.display()

Q2.write.mode("overwrite").saveAsTable("Weather_Analytics.gold.gold_heatwave_alerts")

city,heatwave_occurrences,first_occurrence,latest_occurrence,alert_status
Delhi,28,2026-03-01T04:00:00.000Z,2026-03-08T12:00:00.000Z,Active
Chennai,12,2026-03-01T00:00:00.000Z,2026-03-05T11:00:00.000Z,Monitoring
Hyderabad,4,2026-03-05T01:00:00.000Z,2026-03-05T08:00:00.000Z,Monitoring
Bangalore,3,2026-03-05T01:00:00.000Z,2026-03-05T10:00:00.000Z,Monitoring
Pune,4,2026-03-05T02:00:00.000Z,2026-03-05T10:00:00.000Z,Monitoring
Mumbai,5,2026-03-01T05:00:00.000Z,2026-03-05T11:00:00.000Z,Monitoring


In [0]:
Q3 = spark.sql("""
WITH base AS (
    SELECT
        city,
        event_time,
        rainfall,
        ROW_NUMBER() OVER (PARTITION BY city ORDER BY event_time) -
        ROW_NUMBER() OVER (PARTITION BY city, (rainfall > 0) ORDER BY event_time) AS grp
    FROM Weather_Analytics.silver.silver_weather_clean
),

streaks AS (
    SELECT
        city,
        MIN(event_time) AS streak_start,
        MAX(event_time) AS streak_end,
        COUNT(*) AS streak_hours
    FROM base
    WHERE rainfall > 0
    GROUP BY city, grp
)

SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY streak_hours DESC) AS rn
    FROM streaks
) t
WHERE rn = 1
""")

Q3.display()
Q3.write.mode("overwrite").saveAsTable("Weather_Analytics.gold.gold_rainfall_streaks")

city,streak_start,streak_end,streak_hours,rn
Bangalore,2026-02-22T00:00:00.000Z,2026-03-08T23:00:00.000Z,235,1
Chennai,2026-02-22T00:00:00.000Z,2026-03-07T07:00:00.000Z,194,1
Delhi,2026-02-22T00:00:00.000Z,2026-03-06T14:00:00.000Z,180,1
Hyderabad,2026-02-22T00:00:00.000Z,2026-03-05T04:00:00.000Z,150,1
Mumbai,2026-03-02T13:00:00.000Z,2026-03-08T23:00:00.000Z,152,1
Pune,2026-02-22T00:00:00.000Z,2026-03-06T12:00:00.000Z,178,1


In [0]:
Q4 = spark.sql("""
WITH temp_diff AS (
    SELECT
        city,
        event_time,
        LAG(event_time) OVER (PARTITION BY city ORDER BY event_time) AS previous_event_time,
        LAG(temperature) OVER (PARTITION BY city ORDER BY event_time) AS previous_temperature,
        temperature AS current_temperature
    FROM Weather_Analytics.silver.silver_weather_clean
)

SELECT
    city,
    event_time,
    previous_event_time,
    previous_temperature,
    current_temperature,
    ABS(current_temperature - previous_temperature) AS difference

FROM temp_diff

WHERE previous_temperature IS NOT NULL
  AND ABS(current_temperature - previous_temperature) > 8
""")

Q4.display()

Q4.write.mode("overwrite").saveAsTable("Weather_Analytics.gold.gold_temperature_change")

city,event_time,previous_event_time,previous_temperature,current_temperature,difference
Bangalore,2026-02-22T15:00:00.000Z,2026-02-22T14:00:00.000Z,30.1,21.7,8.400000000000002
Bangalore,2026-02-23T18:00:00.000Z,2026-02-23T17:00:00.000Z,31.7,22.9,8.8
Bangalore,2026-03-01T07:00:00.000Z,2026-03-01T06:00:00.000Z,20.1,29.7,9.599999999999998
Bangalore,2026-03-01T23:00:00.000Z,2026-03-01T22:00:00.000Z,21.1,30.2,9.099999999999998
Bangalore,2026-03-02T02:00:00.000Z,2026-03-02T01:00:00.000Z,21.1,30.7,9.599999999999998
Bangalore,2026-03-02T04:00:00.000Z,2026-03-02T03:00:00.000Z,29.8,20.3,9.5
Bangalore,2026-03-02T09:00:00.000Z,2026-03-02T08:00:00.000Z,31.6,23.2,8.400000000000002
Bangalore,2026-03-03T12:00:00.000Z,2026-03-03T11:00:00.000Z,29.6,21.5,8.100000000000001
Bangalore,2026-03-03T16:00:00.000Z,2026-03-03T15:00:00.000Z,20.5,29.3,8.8
Bangalore,2026-03-03T22:00:00.000Z,2026-03-03T21:00:00.000Z,21.3,30.8,9.5


In [0]:
Q5 = spark.sql("""
SELECT
    b.data_provider,
    COUNT(*) AS total_records,
    COUNT(q.data_provider) AS quarantined_records,
    0 AS duplicate_records,

    ROUND(
        100 - (COUNT(q.data_provider) / COUNT(*) * 100), 2
    ) AS quality_score

FROM Weather_Analytics.bronze.bronze_weather b
LEFT JOIN Weather_Analytics.silver.silver_weather_quarantine q
ON b.data_provider = q.data_provider

GROUP BY b.data_provider
ORDER BY quality_score ASC
""")

Q5.display()
Q5.write.mode("overwrite").saveAsTable("Weather_Analytics.gold.gold_provider_quality")

data_provider,total_records,quarantined_records,duplicate_records,quality_score
AtmosAPI,7676,7676,0,0.0
SkyWatch,11900,11900,0,0.0
WeatherPro,456,0,0,100.0
ClimaData,420,0,0,100.0
